# Notebook 4: EviDTI Model Training and Evaluation (DrugBank)

**Purpose:** Train and evaluate the reproduced EviDTI model on the DrugBank dataset.

**Inputs (pre-extracted in Notebooks 1-3):**
- `drug_embeddings_pooled.npy` — 2D drug features from MG-BERT (256-dim, mean-pooled)
- `bisai_HY_QS_3D_emb_64.npy` — 3D drug features from GeoGNN (64-dim)
- `EviDTI_Protein_Features_Unpooled/` — per-protein residue features from ProtT5 (L x 1024)

**Output:** Test set evaluation metrics (Accuracy, Precision, Recall, MCC, F1, AUC, AUPR) and `EviDTI_Phase2_Predictions_Detailed.csv` with per-sample probability and uncertainty scores.

**Reproducibility:** `np.random.seed(1)` is applied to reproduce the train/val/test split, consistent with the original paper. No `torch.manual_seed` is set, matching the original codebase.

**Patches applied:** The original EviDTI codebase requires several modifications to work with pre-extracted `.npy` feature files. All patches are applied in Step 2 before training.

## Step 0: Mount Google Drive and Restore Project Files

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

# Restore full project (contains all pre-extracted features)
zip_path = '/content/drive/MyDrive/EviDTI_Full_Project_Backup.zip'
shutil.unpack_archive(zip_path, '/content/EviDTI-main')

# Copy protein features from Google Drive to local disk for faster I/O during training
os.makedirs('/content/protein_local', exist_ok=True)
print('Copying protein features to local disk (takes ~3-5 min)...')
!cp -r /content/drive/MyDrive/EviDTI_Protein_Features_Unpooled/* /content/protein_local/
print(f"Protein files copied: {len(os.listdir('/content/protein_local'))}")


## Step 1: Install Dependencies

In [ ]:
!pip install rdkit yacs torch-geometric -q
print('Dependencies installed.')


## Step 2: Apply Source Code Patches

**Patch 2a (`function.py`, `drugbank_model.py`, `main_drugbank.py`):**
Rewrites `bond_angle_graph_data` from `InMemoryDataset` to a lazy-loading `PyGDataset`.
The new class handles variable-length protein sequences with fixed-length padding (max 1000 residues),
attaches drug/protein IDs for downstream analysis, and updates data file paths.
Also fixes PyG `Batch` collation of bag sub-graphs in `drugbank_model.py`.

**Patch 2b (`function.py`):** Adds `unsqueeze(0)` to protein tensor.

**Patch 2c (`function.py`):** Adds `view(1,-1)` to drug 2D feature tensor.

**Patch 2d (`drugbank_solver.py`, `drugbank_model.py`):**
- A: `loss.mean()` instead of `loss.sum()`
- B: KL annealing + scheduler step
- C: Fix belief mass formula Eq.(10)
- D: CosineAnnealingLR T_max 10 → 30, eta_min=1.5e-4
- E: Label dtype float16 → float32
- node_offset fix for graph batching

**Patch 2e (`drugbank_model.py`):** Guard against 1D edge attributes.

**Patch 2f (`drugbank_model.py`):** Fix Light Attention mask alignment.

**Patch 2g (`drugbank_solver.py`):** Inject val_loss + TensorBoard dual-track Loss + CSV logging.

In [ ]:
import os
import re

# ==========================================
# 1. Rewrite function.py (lazy loading + fixed tensor dimensions + ID attachment)
# ==========================================
import re

f_path = '/content/EviDTI-main/function.py'
with open(f_path, 'r', encoding='utf-8') as f:
    content = f.read()

new_class = """
from torch_geometric.data import Dataset as PyGDataset

class bond_angle_graph_data(PyGDataset):
    def __init__(self, root, d_2D_path, t_1D_path, d_3D_path, label_file, metadata_path,
                 transform=None, pre_transform=None, pre_filter=None):
        self.d_2D_path = d_2D_path
        self.t_1D_path = t_1D_path
        self.d_3D_path = d_3D_path
        self.label_file = label_file
        self.metadata_path = metadata_path

        super(bond_angle_graph_data, self).__init__(root, transform, pre_transform, pre_filter)

        import pandas as pd
        import numpy as np
        import os

        print("Initialising EviDTI dataset (fixed-length padding + auto-stacking + ID attachment)...")
        self.meta_df = pd.read_csv(metadata_path)
        self.labels = pd.read_csv(label_file).iloc[:, 2].values

        d_2D_raw = np.load(d_2D_path, allow_pickle=True)
        self.d_2D_dict = d_2D_raw.item() if d_2D_raw.ndim == 0 else d_2D_raw

        d_3D_raw = np.load(d_3D_path, allow_pickle=True)
        if d_3D_raw.ndim == 1:
            self.d_3D_dict = {item['smiles']: item for item in d_3D_raw if 'smiles' in item}
        else:
            self.d_3D_dict = d_3D_raw.item()

        self.valid_indices = []
        for idx in range(len(self.meta_df)):
            smile = self.meta_df.iloc[idx]['smile']
            uid = self.meta_df.iloc[idx]['uid']
            if smile in self.d_3D_dict and smile in self.d_2D_dict:
                if os.path.exists(os.path.join(self.t_1D_path, f"{uid}.npy")):
                    self.valid_indices.append(idx)
        print(f"Dataset initialised. Valid samples: {len(self.valid_indices)}")

    @property
    def raw_file_names(self): return []
    @property
    def processed_file_names(self): return []

    def len(self):
        return len(self.valid_indices)

    def get(self, idx):
        import torch
        import numpy as np
        import os
        from torch_geometric.data import Data

        real_idx = self.valid_indices[idx]
        row = self.meta_df.iloc[real_idx]
        current_uid = row['uid']
        current_smile = row['smile']
        label = self.labels[real_idx]

        # 1. Truncate/pad protein sequence to fixed length 1000
        prot_path = os.path.join(self.t_1D_path, f"{current_uid}.npy")
        t_1D_fea = np.load(prot_path, allow_pickle=True)
        if hasattr(t_1D_fea, 'is_sparse') and t_1D_fea.is_sparse: t_1D_fea = t_1D_fea.to_dense()
        if t_1D_fea.ndim == 1: t_1D_fea = t_1D_fea[np.newaxis, :]

        L = t_1D_fea.shape[0]
        actual_len = min(L, 1000)

        if L >= 1000:
            t_1D_fea = t_1D_fea[:1000, :]
        else:
            t_1D_fea = np.pad(t_1D_fea, ((0, 1000 - L), (0, 0)), mode='constant')

        # Keep as 2D tensor; DataLoader will stack across the batch dimension
        t_tensor = torch.tensor(t_1D_fea, dtype=torch.float)

        # 2. Drug features (stack with explicit axis)
        d_3D_info = self.d_3D_dict[current_smile]

        atom_feature = np.stack([
            d_3D_info['atomic_num'], d_3D_info['chiral_tag'], d_3D_info['degree'],
            d_3D_info['explicit_valence'], d_3D_info['formal_charge'],
            d_3D_info['hybridization'], d_3D_info['implicit_valence']
        ], axis=1)

        bond_feature = np.stack([
            d_3D_info['bond_dir'], d_3D_info['bond_type'],
            d_3D_info['is_in_ring'], d_3D_info['bond_length']
        ], axis=1)

        angle_feature = np.array(d_3D_info['bond_angle'])
        if angle_feature.ndim == 1:
            angle_feature = np.expand_dims(angle_feature, axis=1)

        # 3. Assemble PyG graphs
        atom_bond_graph = Data(
            x=torch.tensor(atom_feature, dtype=torch.float),
            edge_index=torch.tensor(d_3D_info['edges'], dtype=torch.long).t().contiguous(),
            edge_attr=torch.tensor(bond_feature, dtype=torch.float)
        )

        bag_graph = Data(
            x=torch.tensor(bond_feature, dtype=torch.float),
            edge_index=torch.tensor(d_3D_info['BondAngleGraph_edges'], dtype=torch.long).t().contiguous(),
            edge_attr=torch.tensor(angle_feature, dtype=torch.float)
        )
        atom_bond_graph.bag = bag_graph

        # 4. Global features and label
        d_2D_fea = np.array(self.d_2D_dict[current_smile]).flatten()
        atom_bond_graph.d_2D_feature = torch.tensor(d_2D_fea, dtype=torch.float)
        atom_bond_graph.t_1D_feature = t_tensor
        atom_bond_graph.l = torch.tensor(label, dtype=torch.long)

        # Attach IDs as top-level attributes for downstream uncertainty analysis
        atom_bond_graph.cid = str(row.get('cid', current_smile))
        atom_bond_graph.unid = str(current_uid)

        atom_bond_graph.metadata = {
            'id': f"{current_uid}_{current_smile}",
            'd_id': current_smile,
            't_id': current_uid,
            'smile': current_smile,
            'length': torch.tensor(actual_len, dtype=torch.long)
        }

        return atom_bond_graph
"""
pattern = r'class bond_angle_graph_data\(InMemoryDataset\):.*?torch\.save\(\(data_1, slices_1\), self\.processed_paths\[0\]\)'
content = re.sub(pattern, new_class, content, flags=re.DOTALL)
with open(f_path, 'w', encoding='utf-8') as f:
    f.write(content)
print("Patch 2a complete: function.py rewritten, drugbank_model.py and main_drugbank.py updated.")

# ==========================================
# 2. Fix drugbank_model.py (intercept PyG Batch collation of bag sub-graphs)
# ==========================================
m_path = '/content/EviDTI-main/drugbank_model.py'
with open(m_path, 'r', encoding='utf-8') as f:
    m_content = f.read()

bag_fix = """    def forward(self, data):
        # Re-pack bag list into a valid PyG Batch after DataLoader collation
        if hasattr(data, 'bag') and isinstance(data.bag, list):
            from torch_geometric.data import Batch
            data.bag = Batch.from_data_list(data.bag).to(data.x.device)
"""
if "from torch_geometric.data import Batch" not in m_content:
    m_content = re.sub(r'    def forward\(self, data\):', bag_fix, m_content, count=1)
    with open(m_path, 'w', encoding='utf-8') as f:
        f.write(m_content)

# ==========================================
# 3. Fix main_drugbank.py (update data paths to pre-extracted feature files)
# ==========================================
main_path = '/content/EviDTI-main/main_drugbank.py'
with open(main_path, 'r', encoding='utf-8') as f:
    main_content = f.read()

main_content = re.sub(r"from function import bond_angle_graph_data\s*,\s*KIBA_graph_data\s*,\s*case_graph_data",
                   "from function import bond_angle_graph_data", main_content)

old_data_code = """    data_list = bond_angle_graph_data(root='./dataset/bond_angle/',
                                      t_1D_path='dataset/bond_angle/t_feature.npy',
                                      d_2D_path='dataset/bond_angle/d_feature.npy',
                                      d_3D_path='dataset/bond_angle/d_3D_feature.npy',
                                      label_file='dataset/total_cid_unid_csv.csv',
                                      metadata_path='dataset/bond_angle/metadata.csv')"""

new_data_code = """    data_list = bond_angle_graph_data(root='/content/EviDTI-main/dataset/drugbank_dataset/',
                                      t_1D_path='/content/protein_local',
                                      d_2D_path='/content/EviDTI-main/drug_embeddings_pooled.npy',
                                      d_3D_path='/content/EviDTI-main/bisai_HY_QS_3D_emb_64.npy',
                                      label_file='/content/EviDTI-main/dataset/drugbank_dataset/total_cid_unid_csv.csv',
                                      metadata_path='/content/EviDTI-main/dataset/drugbank_dataset/metadata.csv')"""

main_content = main_content.replace(old_data_code, new_data_code)
with open(main_path, 'w', encoding='utf-8') as f:
    f.write(main_content)



In [ ]:
import os

f_path = '/content/EviDTI-main/function.py'
with open(f_path, 'r', encoding='utf-8') as f:
    content = f.read()

# Add unsqueeze(0) to restore the batch dimension expected by downstream layers
old_code = "t_tensor = torch.tensor(t_1D_fea, dtype=torch.float)"


new_code = "t_tensor = torch.tensor(t_1D_fea, dtype=torch.float).unsqueeze(0)"

if old_code in content:
    content = content.replace(old_code, new_code)
    with open(f_path, 'w', encoding='utf-8') as f:
        f.write(content)
    print("Patch 2b applied: protein tensor unsqueeze(0).")
else:
    print("Patch 2b: target line not found, skipping.")

In [ ]:
import os

f_path = '/content/EviDTI-main/function.py'
with open(f_path, 'r', encoding='utf-8') as f:
    content = f.read()

# Add view(1,-1) to promote drug 2D feature from (256,) to (1,256)
old_str = "atom_bond_graph.d_2D_feature = torch.tensor(d_2D_fea, dtype=torch.float)"


new_str = "atom_bond_graph.d_2D_feature = torch.tensor(d_2D_fea, dtype=torch.float).view(1, -1)"

if old_str in content:
    content = content.replace(old_str, new_str)
    with open(f_path, 'w', encoding='utf-8') as f:
        f.write(content)
    print("Patch 2c applied: drug 2D feature view(1,-1).")
else:
    print("Patch 2c: target line not found, skipping.")

In [ ]:
import os
import shutil

def final_consolidated_fix(base_path="/content/EviDTI-main"):
    print("="*60)
    print(f"Applying patches to: {base_path}")
    print("="*60)

    solver_path = os.path.join(base_path, 'drugbank_solver.py')
    model_path = os.path.join(base_path, 'drugbank_model.py')

    if not os.path.exists(solver_path) or not os.path.exists(model_path):
        print(f"Error: target files not found in {base_path}.")
        return

    for f_path in [solver_path, model_path]:
        bak_path = f_path + '.master_bak'
        if not os.path.exists(bak_path):
            shutil.copy(f_path, bak_path)
            print(f"Backup created: {os.path.basename(bak_path)}")

    # ==========================================
    # Fix drugbank_solver.py
    # ==========================================
    with open(solver_path, 'r', encoding='utf-8') as f:
        s_code = f.read()

    # A: loss.mean() + inject lam parameter
    s_code = s_code.replace("loss = loss.sum()", "loss = loss.mean()")
    if "lam: float = 0.1" not in s_code:
        s_code = s_code.replace(
            "def predict(self, data_loader: DataLoader, epoch: int = None, optim: torch.optim.Optimizer = None) -> \\",
            "def predict(self, data_loader: DataLoader, epoch: int = None, optim: torch.optim.Optimizer = None, lam: float = 0.1) -> \\"
        )
    s_code = s_code.replace("lam=0.1", "lam=lam")

    # B: KL annealing + scheduler step
    if "current_lam =" not in s_code:
        s_code = s_code.replace(
            "for epoch in range(self.start_epoch, self.epoch):",
            "for epoch in range(self.start_epoch, self.epoch):\n            current_lam = min(0.3, (epoch + 1) / 50.0)"
        )
        s_code = s_code.replace(
            "train_loss, train_results = self.predict(train_loader, epoch + 1, optim=self.optim)",
            "train_loss, train_results = self.predict(train_loader, epoch + 1, optim=self.optim, lam=current_lam)"
        )
    if "self.scheduler.step()" not in s_code:
        s_code = s_code.replace(
            "if epochs_no_improve >= 30 and max_train_acc >= min_train_acc:",
            "self.scheduler.step()\n            if epochs_no_improve >= 30 and max_train_acc >= min_train_acc:"
        )

    # C: Fix belief mass formula Eq.(10)
    s_code = s_code.replace(
        'bk = alphas -1 / np.sum(alphas, axis=-1)',
        'bk = (alphas - 1) / (np.sum(alphas, axis=-1, keepdims=True) + 1e-9)'
    )

    # D: CosineAnnealingLR T_max 10 -> 30
    s_code = s_code.replace(
        'CosineAnnealingLR(self.optim, T_max=10)',
        'CosineAnnealingLR(self.optim, T_max=30, eta_min=1.5e-4)'
    )

    # E: label dtype float16 -> float32
    s_code = s_code.replace(
        'label.to(torch.float16)',
        'label.to(torch.float32)'
    )

    with open(solver_path, 'w', encoding='utf-8') as f:
        f.write(s_code)

    assert 'T_max=30' in s_code,    "Patch D failed: T_max not replaced"
    assert 'float16' not in s_code, "Patch E failed: float16 still present"
    assert 'float32' in s_code,     "Patch E failed: float32 not written"
    print("drugbank_solver.py patches A-E applied.")
    print("  D: CosineAnnealingLR T_max=10 -> 30")
    print("  E: label float16 -> float32")

    # ==========================================
    # Fix drugbank_model.py (node_offset for graph batching)
    # ==========================================
    with open(model_path, 'r', encoding='utf-8') as f:
        m_lines = f.readlines()

    new_m_lines = []
    for i, line in enumerate(m_lines):
        if 'j = 0' in line and 'node_offset' not in line:
            line = line.replace('j = 0', 'j = 0\n        node_offset = 0')
        if 'g_edge_index = g.edge_index' in line and '+ node_offset' not in line:
            line = line.replace('g.edge_index', 'g.edge_index + node_offset')
        if 'j += 1' in line and 'node_offset +=' not in line:
            line = '            node_offset += g_x.size(0)\n' + line
        if 'self.bag_conv3(' in line:
            chunk = "".join(m_lines[max(0, i-5):i])
            if '#5' in chunk:
                line = line.replace('self.bag_conv3(', 'self.bag_conv5(')
            elif '#6' in chunk:
                line = line.replace('self.bag_conv3(', 'self.bag_conv6(')
        new_m_lines.append(line)

    with open(model_path, 'w', encoding='utf-8') as f:
        f.writelines(new_m_lines)
    print("drugbank_model.py node_offset fix applied.")
    print("="*60)
    print("Patch 2d complete. Please restart the Colab runtime before training.")

# Run all fixes
final_consolidated_fix()

In [ ]:
import os
import re

m_path = '/content/EviDTI-main/drugbank_model.py'
with open(m_path, 'r', encoding='utf-8') as f:
    content = f.read()

# Guard against 1D edge_attr_list for small graphs
pattern = r"([ \t]+)edge_int_feat = edge_attr_list\[:, 0:3\].to\('cpu'\).to\(torch\.int64\)"


def replacer(match):
    indent = match.group(1)
    return (f"{indent}# Ensure edge_attr_list is 2D before column slicing\n"
            f"{indent}if hasattr(edge_attr_list, 'dim') and edge_attr_list.dim() == 1:\n"
            f"{indent}    edge_attr_list = edge_attr_list.view(-1, 4) if edge_attr_list.numel() % 4 == 0 else edge_attr_list.view(-1, 1)\n"
            f"{indent}edge_int_feat = edge_attr_list[:, 0:3].to('cpu').to(torch.int64)")

new_content, count = re.subn(pattern, replacer, content)

if count > 0:
    with open(m_path, 'w', encoding='utf-8') as f:
        f.write(new_content)
    print(f"Patch 2e applied: edge_attr_list 1D guard ({count} location(s)).")
else:
    print("Patch 2e: target line not found, skipping.")

In [ ]:
import os

m_path = '/content/EviDTI-main/drugbank_model.py'
with open(m_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

# Step 1: Remove previously injected (malformed) mask alignment code
cleaned_lines = []
skip = False
for line in lines:
    if "# Ensure edge_attr_list is 2D" in line:
        skip = True
        continue
    if skip:
        # Found original line, stop skipping
        if "attention = attention.masked_fill(" in line:
            skip = False
            # Restore original line with correct indentation
            cleaned_lines.append("        attention = attention.masked_fill(mask[:, None, :] == False, -1e9)\n")
        continue
    cleaned_lines.append(line)

# Step 2: Re-inject mask alignment with correct indentation
final_lines = []
for line in cleaned_lines:
    if "attention = attention.masked_fill(mask[:, None, :] == False, -1e9)" in line:
        final_lines.extend([
            "        # Align mask length to attention size for variable-length sequences\n",
            "        if mask is not None and mask.size(-1) != attention.size(-1):\n",
            "            import torch.nn.functional as F\n",
            "            if mask.size(-1) < attention.size(-1):\n",
            "                mask = F.pad(mask, (0, attention.size(-1) - mask.size(-1)), value=False)\n",
            "            else:\n",
            "                mask = mask[:, :attention.size(-1)]\n",
            "        attention = attention.masked_fill(mask[:, None, :] == False, -1e9)\n"
        ])
    else:
        final_lines.append(line)

with open(m_path, 'w', encoding='utf-8') as f:
    f.writelines(final_lines)

print("Patch 2f applied: Light Attention mask alignment fix.")

In [ ]:
import os

# 1. Locate solver file
solver_file = "/content/EviDTI-main/drugbank_solver.py"

if os.path.exists(solver_file):
    print(f"Applying Patch 2g to {solver_file}...")

    with open(solver_file, "r", encoding="utf-8") as f:
        lines = f.readlines()

    new_lines = []
    skip_next_line = False

    for line in lines:
        if skip_next_line:
            skip_next_line = False
            continue

        # F: inject val_loss computation (missing in original codebase, needed for early stopping)
        if "_,_,val_results,c,var,prob_list,_,_ = self.predict_val(val_loader, epoch + 1)" in line:
            indent = line[:line.find("_")]
            new_lines.append(f"{indent}val_loss, _ = self.predict(val_loader, epoch + 1, optim=None)\n")
            new_lines.append(line)
            continue

        # Compatibility: extract val_loss from predict() if called directly
        if "val_results = self.predict(" in line:
            line = line.replace("val_results =", "val_loss, val_results =")
            new_lines.append(line)
            continue

        # G: remove single-track Loss scalar
        if "self.writer.add_scalar('Loss/train'" in line:
            continue

        # G: inject dual-track Train/Val Loss into TensorBoard
        if "self.writer.add_scalars('MCC'" in line:
            new_lines.append(line)
            indent = line[:line.find("self.writer")]
            new_lines.append(f"{indent}self.writer.add_scalars('Loss', {{'Train': train_loss, 'Val': val_loss}}, epoch + 1)\n")
            continue

        # G: update epoch print and add per-epoch CSV logging
        if "print('[Epoch %d] Loss:" in line or "print('[Epoch %d] val accuracy:" in line:
            indent = line[:line.find("print")]

            # Single-line epoch summary
            new_lines.append(f"{indent}print('[Epoch %d] Train Loss: %.4f | Val Loss: %.4f | Train Acc: %.2f%% | Val Acc: %.2f%% | Val MCC: %.4f' % (epoch + 1, train_loss, val_loss, train_acc, val_acc, val_mcc))\n")

            # Get save path dynamically from solver attributes
            new_lines.append(f"{indent}save_path = getattr(self, 'result_dir', getattr(self, 'save_dir', self.writer.log_dir))\n")
            new_lines.append(f"{indent}import pandas as pd\n")
            new_lines.append(f"{indent}import os as _os\n")

            # Create separate Train/Val subdirectories under runs/
            new_lines.append(f"{indent}train_dir = _os.path.join(save_path, 'train_loss')\n")
            new_lines.append(f"{indent}val_dir = _os.path.join(save_path, 'val_loss')\n")
            new_lines.append(f"{indent}_os.makedirs(train_dir, exist_ok=True)\n")
            new_lines.append(f"{indent}_os.makedirs(val_dir, exist_ok=True)\n")

            # Accumulate per-epoch history
            new_lines.append(f"{indent}if not hasattr(self, 'train_history'): self.train_history = []\n")
            new_lines.append(f"{indent}if not hasattr(self, 'val_history'): self.val_history = []\n")

            new_lines.append(f"{indent}self.train_history.append({{'Epoch': epoch + 1, 'Train_Loss': train_loss, 'Train_Acc': train_acc, 'Train_MCC': train_mcc}})\n")
            new_lines.append(f"{indent}self.val_history.append({{'Epoch': epoch + 1, 'Val_Loss': val_loss, 'Val_Acc': val_acc, 'Val_MCC': val_mcc}})\n")

            # Save history to CSV
            new_lines.append(f"{indent}pd.DataFrame(self.train_history).to_csv(_os.path.join(train_dir, 'train_loss.csv'), index=False)\n")
            new_lines.append(f"{indent}pd.DataFrame(self.val_history).to_csv(_os.path.join(val_dir, 'val_loss.csv'), index=False)\n")

            skip_next_line = True  # skip the second original print line
            continue

        new_lines.append(line)

    with open(solver_file, "w", encoding="utf-8") as f:
        f.writelines(new_lines)

    print("Patch 2g applied: val_loss injection + TensorBoard dual-track Loss + CSV logging.")
else:
    print("Patch 2g: solver file not found.")

## Step 3: Train the Model

In [ ]:
# Clear any previous run artifacts before training
!rm -rf /content/EviDTI-main/runs/*

# Launch training using the original EviDTI training script.
# Key settings:
#   Optimizer: Adam, lr=3e-4, weight_decay=1.5e-5
#   Batch size: 64 | Epochs: 30 
#   Train/val/test split: random 8:1:1
# Best model checkpoint is saved automatically under runs/
%cd /content/EviDTI-main
!python main_drugbank.py --cfg configs/drugbank.yaml --data drugbank --split random


## Step 4: Save Training Results to Google Drive

In [ ]:
backup_dir = '/content/drive/MyDrive/EviDTI_Seed_1_Train_Results_Backup/'
!mkdir -p {backup_dir}
!cp -r /content/EviDTI-main/runs {backup_dir} 2>/dev/null || true
!cp -r /content/EviDTI-main/result {backup_dir} 2>/dev/null || true
!cp /content/EviDTI-main/*.csv {backup_dir} 2>/dev/null || true
!cp /content/EviDTI-main/*.pt {backup_dir} 2>/dev/null || true
print(f'Training results backed up to: {backup_dir}')


## Step 5: Evaluate on Test Set

In [ ]:
# EviDTI test set evaluation
import sys
import os
import torch
import numpy as np
from torch_geometric.loader import DataLoader
from torch.utils.data import SubsetRandomSampler
np.random.seed(1)

# 1. Set working directory
os.chdir('/content/EviDTI-main')
if '/content/EviDTI-main' not in sys.path:
    sys.path.append('/content/EviDTI-main')

from drugbank_configs import get_cfg_defaults
from drugbank_model import LightAttention
from drugbank_solver import Solver
from function import bond_angle_graph_data

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Load config
print("Loading config...")
cfg = get_cfg_defaults()

# 3. Load dataset and reproduce the same split used during training
print("Loading dataset...")
data_list = bond_angle_graph_data(root='/content/EviDTI-main/dataset/drugbank_dataset/',
                                  t_1D_path='/content/protein_local',
                                  d_2D_path='/content/EviDTI-main/drug_embeddings_pooled.npy',
                                  d_3D_path='/content/EviDTI-main/bisai_HY_QS_3D_emb_64.npy',
                                  label_file='/content/EviDTI-main/dataset/drugbank_dataset/total_cid_unid_csv.csv',
                                  metadata_path='/content/EviDTI-main/dataset/drugbank_dataset/metadata.csv')

shuffled_indices = np.random.permutation(len(data_list))
test_idx = shuffled_indices[int(0.9 * len(data_list)):]
test_loader = DataLoader(data_list, batch_size=cfg.SOLVER.BATCH_SIZE, drop_last=False, sampler=SubsetRandomSampler(test_idx))
print(f"Test set size: {len(test_idx)}")

# 4. Build model and load checkpoint
print("Building model...")
model = LightAttention(cfg).to(device)

model_path = "/content/EviDTI-main/runs/la_test1_31-03_02-31-10/checkpoint.pth"
print(f"Loading checkpoint: {model_path}")

if os.path.exists(model_path):
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)

    if isinstance(checkpoint, dict):
        if 'state_dict' in checkpoint:
            model.load_state_dict(checkpoint['state_dict'])
        else:
            model.load_state_dict(checkpoint)
    elif isinstance(checkpoint, torch.nn.Module):
        model.load_state_dict(checkpoint.state_dict())

    print("Checkpoint loaded.")
else:
    print(f"Checkpoint not found: {model_path}")

print("\nPhase 1: Running official evaluation...")
model.eval()
solver = Solver(model, cfg, device)

try:
    # Phase 1: official evaluation (macro metrics)
    solver.evaluation(test_loader)
    print("Phase 1 complete.")

    # Phase 2: extract per-sample probability and uncertainty
    print("\nPhase 2: Extracting predictions and uncertainties...")
    model.eval()
    results_list = []

    with torch.no_grad():
        for batch in test_loader:

            # Move metadata to GPU manually (do not call batch.to(device))
            metadata = batch.metadata
            sequence_lengths = metadata['length'][:, None].to(device)

            # Build attention mask
            mask = torch.arange(metadata['length'].max())[None, :] < metadata['length'][:, None]
            mask = mask.to(device)

            # Forward pass
            outputs = model(batch, mask=mask, sequence_lengths=sequence_lengths)

            # Extract labels and IDs
            labels = batch.l.cpu().numpy()
            cids = batch.cid
            unids = batch.unid

            # Compute probability and evidential uncertainty (u = K/sum(alpha), K=2)
            S = torch.sum(outputs, dim=1, keepdim=True)
            probs = (outputs[:, 1:2] / S).cpu().numpy().flatten()
            uncertainties = (2.0 / S).cpu().numpy().flatten()

            # Append to results
            for i in range(len(labels)):
                results_list.append({
                    'Drug_CID': cids[i],
                    'Target_Uniprot': unids[i],
                    'True_Label': labels[i],
                    'Predicted_Probability': probs[i],
                    'Uncertainty': uncertainties[i]
                })

    # Export predictions to CSV
    import pandas as pd
    df_results = pd.DataFrame(results_list)
    output_csv_path = 'EviDTI_Phase2_Predictions_Detailed.csv'
    df_results.to_csv(output_csv_path, index=False)

    print(f"Saved {len(df_results)} predictions to {output_csv_path}")
    print("\nPreview:")
    print(df_results.head())

except Exception as e:
    print(f"Error during evaluation: {e}")

## Step 6: Save Evaluation Results to Google Drive

In [ ]:
import shutil

backup_dir = '/content/drive/MyDrive/EviDTI_Seed_1_Test_Results_Backup'
os.makedirs(backup_dir, exist_ok=True)
shutil.copy('EviDTI_Phase2_Predictions_Detailed.csv', backup_dir)
print(f'Evaluation results saved to: {backup_dir}')
